# 🚲 Bike Share Data Preprocessing (US09)

Transforms raw Bike Share Toronto ridership data from the **Bronze** layer into a clean, standardised **Silver** dataset ready for feature engineering and model training.

| Layer | Path | Description |
|---|---|---|
| **Bronze** | `data/bronze/bikeshare_ridership` | Raw Parquet as ingested |
| **Silver** | `data/silver/bikeshare_trips` | Cleaned, partitioned by year/month |

| Step | Purpose |
|---|---|
| 0 | Define paths and business rules |
| 1 | Load Bronze data |
| 2 | Standardise column names to snake_case |
| 3 | Filter null critical fields |
| 4 | Extract day and hour buckets from raw strings |
| 5 | Parse trip duration and remove outliers |
| 6 | De-duplicate by `trip_id` |
| 7 | Build Silver dataset |
| 8 | Write Silver Parquet partitioned by year/month |
| 9 | Validate with summary statistics |


## Steps 0–9 — Full Preprocessing Pipeline

**Step 0 — Config:** sets Bronze/Silver DBFS paths and the business rule cap of `MAX_DURATION_MIN = 240` (4 hours) to exclude operational anomalies.

**Step 1 — Load Bronze:** reads the raw ridership Parquet and prints the original schema for reference.

**Step 2 — Standardise column names:** converts all column headers to lowercase snake_case (e.g. `"Trip  Duration"` → `"trip_duration"`). Start/end times and duration are cast to strings to prevent implicit timestamp casting issues in Databricks Serverless.

**Step 3 — Null filtering:** removes any row missing a `trip_id`, `start_time`, `end_time`, `start_station_id`, `end_station_id`, `year`, or `month`. These are the minimum fields required for downstream joins and aggregations.

**Step 4 — Extract day and hour:** since Databricks Serverless does not reliably support `to_timestamp()` / `unix_timestamp()`, times are parsed via string splits. Two observed formats (`"7/1/2024 0:00"` and `"07/01/2023 00:00"`) are handled. The resulting `hour_start_str` and `hour_end_str` use the `HH:00` format consistent with the weather and events layers. Rows with out-of-range day (1–31) or hour (0–23) values are dropped.

**Step 5 — Trip duration:** the raw duration value is extracted as a number, interpreted as **seconds**, and converted to **minutes**. Trips with zero or negative duration and trips exceeding 240 minutes are removed as unrealistic (e.g. bikes left undocked or system errors).

**Step 6 — De-duplication:** removes exact duplicate `trip_id` entries, keeping the first occurrence.

**Step 7 — Build Silver:** selects only the columns needed downstream: identifiers, station info, time dimensions, duration, and source file.

**Step 8 — Write Silver:** writes the clean dataset to DBFS as Parquet partitioned by `year` and `month` for efficient downstream filtering.

**Step 9 — Validation:** prints total row count, duration summary statistics (min/max/median/percentiles), duration buckets (0–1h, 1–2h, 2–4h), and row counts by year and year/month to confirm coverage and data quality.


In [0]:
# =========================================
# US09 - Bike Share Preprocessing (Bronze -> Silver)
# Databricks Serverless / UC Volume safe
# Granularity: hour ("HH:00"), floor for both start and end time
# =========================================

from pyspark.sql import functions as F
import re

# ---------------------------
# Paths (adjust only if your team changes the project folder)
# ---------------------------
BRONZE_DIR = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/bronze/bikeshare_ridership"
SILVER_DIR = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/silver/bikeshare_trips"

# Business rule: remove unrealistic trips (operational anomalies)
MAX_DURATION_MIN = 240  # 4 hours

print("=== Bike Share Preprocessing ===")
print("BRONZE:", BRONZE_DIR)
print("SILVER:", SILVER_DIR)

# ---------------------------
# 1) Load Bronze (fresh read to avoid carrying old transformations)
# ---------------------------
df_bronze = spark.read.parquet(BRONZE_DIR)

print("Bronze schema (original):")
df_bronze.printSchema()

# ---------------------------
# 2) Standardize column names to snake_case
#    Example: "Trip  Duration" -> "trip_duration"
# ---------------------------
def to_snake(name: str) -> str:
    s = name.strip().lower()
    s = re.sub(r"[^a-z0-9]+", "_", s)   # replace spaces/symbols by underscore
    s = re.sub(r"_+", "_", s).strip("_")
    return s

df = df_bronze
for c in df_bronze.columns:
    df = df.withColumnRenamed(c, to_snake(c))

print("Columns after rename:")
print(df.columns)

# Ensure start/end times are treated as strings (avoid implicit timestamp casting)
df = (df
      .withColumn("start_time", F.col("start_time").cast("string"))
      .withColumn("end_time",   F.col("end_time").cast("string"))
      .withColumn("trip_duration", F.col("trip_duration").cast("string"))
)

# ---------------------------
# 3) Basic null checks (keep only rows needed for modeling/integration)
# ---------------------------
df = df.filter(
    F.col("trip_id").isNotNull() &
    F.col("start_time").isNotNull() &
    F.col("end_time").isNotNull() &
    F.col("start_station_id").isNotNull() &
    F.col("end_station_id").isNotNull() &
    F.col("year").isNotNull() &
    F.col("month").isNotNull()
)

# ---------------------------
# 4) Extract day + hour buckets WITHOUT timestamps (serverless-safe)
#    Start/End formats observed:
#      "7/1/2024 0:00"
#      "07/01/2023 00:00"
#    We parse by string splits (no to_timestamp / unix_timestamp).
# ---------------------------
df = (df
    .withColumn("start_date_part", F.split(F.col("start_time"), " ").getItem(0))
    .withColumn("start_time_part", F.split(F.col("start_time"), " ").getItem(1))
    .withColumn("end_date_part",   F.split(F.col("end_time"), " ").getItem(0))
    .withColumn("end_time_part",   F.split(F.col("end_time"), " ").getItem(1))

    # day from M/d/yyyy -> second element (index 1)
    .withColumn("day", F.split(F.col("start_date_part"), "/").getItem(1).cast("int"))

    # hour from H:mm -> left side (index 0)
    .withColumn("start_hour", F.split(F.col("start_time_part"), ":").getItem(0).cast("int"))
    .withColumn("end_hour",   F.split(F.col("end_time_part"), ":").getItem(0).cast("int"))

    # hour string format HH:00 (granularity: hour)
    .withColumn("hour_start_str", F.format_string("%02d:00", F.col("start_hour")))
    .withColumn("hour_end_str",   F.format_string("%02d:00", F.col("end_hour")))
)

# Sanity filters on extracted fields
df = df.filter(
    F.col("day").between(1, 31) &
    F.col("start_hour").between(0, 23) &
    F.col("end_hour").between(0, 23)
)

# ---------------------------
# 5) Parse duration (seconds -> minutes) + filter outliers
#    Based on your summary: median ~683, max huge => seconds
# ---------------------------
df = df.withColumn(
    "trip_duration_num",
    F.regexp_extract(F.col("trip_duration"), r"([0-9]+(\.[0-9]+)?)", 1).cast("double")
)

df = df.filter(F.col("trip_duration_num").isNotNull())

# Convert seconds -> minutes
df = df.withColumn("trip_duration_min", F.col("trip_duration_num") / 60.0)

# Remove invalid or extreme durations
df = df.filter(
    (F.col("trip_duration_min") > 0) &
    (F.col("trip_duration_min") <= MAX_DURATION_MIN)
)

# ---------------------------
# 6) De-duplicate by trip_id (keep first occurrence)
# ---------------------------
df = df.dropDuplicates(["trip_id"])

# ---------------------------
# 7) Build Silver dataset (clean, standardized, modeling-friendly)
# ---------------------------
df_silver = df.select(
    "trip_id",
    "bike_id",
    "user_type",
    "model",
    "start_station_id",
    "start_station_name",
    "end_station_id",
    "end_station_name",
    "year", "month", "day",
    "hour_start_str",
    "hour_end_str",
    "trip_duration_min",
    "source_file"
)

# ---------------------------
# 8) Write Silver Parquet (partitioned by year/month)
# ---------------------------
(df_silver.write
 .mode("overwrite")
 .partitionBy("year", "month")
 .parquet(SILVER_DIR)
)

print("✅ Silver written to:", SILVER_DIR)

# ---------------------------
# 9) Validations (evidence for Taiga/PR)
# ---------------------------
total_rows = df_silver.count()
print(f"Total rows in Silver: {total_rows:,}")

print("=== Duration summary (minutes) ===")
display(
    df_silver.select("trip_duration_min")
             .summary("count", "min", "25%", "50%", "75%", "max", "mean")
)

print("=== Duration buckets ===")
display(
    df_silver.groupBy(
        F.when(F.col("trip_duration_min") <= 60, "0–1h")
         .when(F.col("trip_duration_min") <= 120, "1–2h")
         .when(F.col("trip_duration_min") <= 240, "2–4h")
         .otherwise(">4h")
         .alias("duration_bucket")
    ).count().orderBy("duration_bucket")
)

print("=== Rows per year ===")
display(df_silver.groupBy("year").count().orderBy("year"))

print("=== Rows per year & month ===")
display(df_silver.groupBy("year", "month").count().orderBy("year", "month"))

print("=== Silver Schema ===")
df_silver.printSchema()

print("=== DONE (US09 Bike Share preprocessing) ===")

# Downstream example:
# df_silver_read = spark.read.parquet(SILVER_DIR)
